# Chapter 01. 베스트셀러 데이터 이해와 기본 전처리

교보문고 베스트셀러 Excel 데이터를 불러와서 필요한 컬럼만 남기고, 결측치·자료형·중복을 확인한 뒤 CSV 파일로 저장합니다.

> 코드마다 주석을 달아서 무엇을 하는지 바로 볼 수 있게 정리했습니다.


## 실습 1. Excel 파일 불러오기


In [ ]:
# pandas 불러오기
# pd라는 짧은 이름으로 pandas를 사용합니다.
import pandas as pd

# Excel 파일의 위치를 문자열로 저장합니다.
# 현재 VS Code 실행 위치가 C:\\dev\\llm-data-analysis-course 이므로 이 경로를 사용합니다.
file_path = "notebooks/book-text-ml/교보문고_종합_베스트셀러_상품리스트.xlsx"

# Excel 파일을 읽어서 df라는 DataFrame에 저장합니다.
df = pd.read_excel(file_path)

# df가 어떤 자료형인지 확인합니다.
print(type(df))

# 데이터 앞의 5행을 확인합니다.
df.head()


## 실습 2. 데이터의 크기와 컬럼 확인하기


In [ ]:
# shape : 전체 행 개수와 컬럼 개수를 확인합니다.
print("데이터 크기:", df.shape)

# columns : 어떤 컬럼들이 있는지 확인합니다.
print("\n컬럼 이름:")
print(df.columns)

# dtypes : 각 컬럼의 자료형을 확인합니다.
print("\n자료형:")
print(df.dtypes)

# isna().sum() : 각 컬럼에 비어 있는 값이 몇 개인지 확인합니다.
print("\n결측치 개수:")
print(df.isna().sum())

# 마지막으로 앞의 5행을 다시 확인합니다.
df.head()


## 실습 3. 분석에 필요한 컬럼만 선택하기


In [ ]:
# 이번 분석에서 사용할 컬럼 이름을 리스트로 만듭니다.
selected_columns = [
    "순위",
    "판매상품 ID",
    "상품명",
    "판매가",
    "인물",
    "출판사",
    "발행(출시)일자",
    "분야",
]

# df에서 필요한 컬럼만 골라서 df_books에 저장합니다.
# copy()를 사용하면 원본 df와 따로 작업할 수 있습니다.
df_books = df[selected_columns].copy()

# 잘 선택되었는지 확인합니다.
print(df_books.columns)
df_books.head()


## 실습 4. 컬럼 이름을 짧게 바꾸기


In [ ]:
# rename()을 이용해 긴 컬럼 이름을 짧게 바꿉니다.
df_books = df_books.rename(columns={
    "판매상품 ID": "판매상품ID",
    "인물": "저자",
    "발행(출시)일자": "발행일",
})

# 컬럼 이름이 바뀌었는지 확인합니다.
print(df_books.columns)


## 실습 5. 결측치 확인하고 채우기


In [ ]:
# 먼저 처리 전 결측치 개수를 확인합니다.
print("처리 전 결측치")
print(df_books.isna().sum())

# 나중에 요약에 사용하기 위해 처리 전 개수를 변수에 저장합니다.
author_missing_before = df_books["저자"].isna().sum()
category_missing_before = df_books["분야"].isna().sum()

# 저자가 비어 있으면 '미상'으로 채웁니다.
df_books["저자"] = df_books["저자"].fillna("미상")

# 분야가 비어 있으면 '미분류'로 채웁니다.
df_books["분야"] = df_books["분야"].fillna("미분류")

# 처리 후 결측치가 줄었는지 다시 확인합니다.
print("\n처리 후 결측치")
print(df_books.isna().sum())


## 실습 6. 판매가를 숫자로 바꾸기


In [ ]:
# 변환 전 판매가 자료형을 확인합니다.
print("변환 전 자료형:", df_books["판매가"].dtype)

# 판매가를 문자열로 바꾼 뒤 쉼표(,)를 제거합니다.
# 예: '16,200' -> '16200'
df_books["판매가"] = df_books["판매가"].astype(str)
df_books["판매가"] = df_books["판매가"].str.replace(",", "", regex=False)

# 문자열이 된 판매가를 실제 숫자로 변환합니다.
# 숫자로 바꿀 수 없는 값은 NaN으로 처리합니다.
df_books["판매가"] = pd.to_numeric(df_books["판매가"], errors="coerce")

# 변환 결과를 확인합니다.
print("변환 후 자료형:", df_books["판매가"].dtype)
print(df_books["판매가"].head())

# 판매가의 기본 통계를 확인합니다.
df_books["판매가"].describe()


## 실습 7. 발행일을 날짜형으로 바꾸기


In [ ]:
# 발행일을 먼저 문자열로 바꿉니다.
df_books["발행일"] = df_books["발행일"].astype(str)

# Excel에서 20260701.0처럼 읽힌 경우 뒤의 .0을 제거합니다.
df_books["발행일"] = df_books["발행일"].str.replace(".0", "", regex=False)

# YYYYMMDD 형식의 문자열을 날짜형으로 변환합니다.
# errors='coerce'는 날짜로 바꿀 수 없는 값을 NaT로 처리합니다.
df_books["발행일"] = pd.to_datetime(
    df_books["발행일"],
    format="%Y%m%d",
    errors="coerce"
)

# 변환 결과를 확인합니다.
print(df_books["발행일"].head())
print("자료형:", df_books["발행일"].dtype)


## 실습 8. 문자열 앞뒤 공백 정리하기


In [ ]:
# 공백을 정리할 문자열 컬럼을 리스트로 만듭니다.
string_columns = ["상품명", "저자", "출판사", "분야"]

# 각 컬럼을 하나씩 꺼내서 앞뒤 공백을 제거합니다.
for column in string_columns:
    df_books[column] = df_books[column].astype(str).str.strip()

# 앞의 5행을 확인합니다.
df_books[string_columns].head()


## 실습 9. 중복 데이터 확인하기


In [ ]:
# duplicated().sum()은 중복된 행의 개수를 셉니다.

# 전체 행이 완전히 같은 중복 개수
row_duplicate_count = df_books.duplicated().sum()

# 판매상품ID가 중복된 개수
product_id_duplicate_count = df_books["판매상품ID"].duplicated().sum()

# 상품명이 중복된 개수
title_duplicate_count = df_books["상품명"].duplicated().sum()

# 결과 출력
print("전체 행 완전 중복:", row_duplicate_count)
print("판매상품ID 중복:", product_id_duplicate_count)
print("상품명 중복:", title_duplicate_count)


## 실습 10. 전처리 결과 최종 확인하기


In [ ]:
# 전체 행과 컬럼 개수 확인
print("데이터 크기:", df_books.shape)

# 컬럼 이름 확인
print("\n컬럼 이름:")
print(df_books.columns)

# 자료형 확인
print("\n자료형:")
print(df_books.dtypes)

# 결측치 확인
print("\n결측치:")
print(df_books.isna().sum())

# 판매가와 발행일이 제대로 변환되었는지 확인
print("\n판매가 앞의 5개:")
print(df_books["판매가"].head())

print("\n발행일 앞의 5개:")
print(df_books["발행일"].head())

# 판매상품ID 중복 개수 확인
print("\n판매상품ID 중복 개수:", product_id_duplicate_count)

# 최종 데이터 앞의 5행 확인
df_books.head()


## 실습 11. 전처리한 데이터를 CSV로 저장하기


In [ ]:
# 저장할 파일 위치를 지정합니다.
output_path = "notebooks/book-text-ml/book_bestseller_clean.csv"

# index=False : DataFrame의 왼쪽 인덱스 번호는 저장하지 않습니다.
# encoding='utf-8-sig' : Excel에서 한글이 깨지는 것을 줄여줍니다.
df_books.to_csv(output_path, index=False, encoding="utf-8-sig")

# 저장 위치를 출력합니다.
print("저장 완료:", output_path)


## 실습 12. 실제 결과를 이용해 간단한 요약 만들기


In [ ]:
# 위에서 직접 계산한 값들을 이용해 결과 문장을 만듭니다.
print("전처리 결과 요약")
print("- 원본 데이터 크기:", df.shape)
print("- 선택한 컬럼 수:", df_books.shape[1])
print("- 처리 전 저자 결측치:", author_missing_before)
print("- 처리 전 분야 결측치:", category_missing_before)
print("- 판매상품ID 중복:", product_id_duplicate_count)
print("- 판매가 자료형:", df_books["판매가"].dtype)
print("- 발행일 자료형:", df_books["발행일"].dtype)


## 확인 문제

**질문 1. 왜 모든 원본 컬럼을 사용하지 않고 필요한 컬럼만 선택했나요?**  
분석에 필요하지 않은 컬럼까지 가지고 있으면 데이터가 복잡해지기 때문에 필요한 정보만 남겼습니다.

**질문 2. 판매가가 문자열이라면 어떤 문제가 발생할 수 있나요?**  
문자열 상태에서는 평균이나 합계 같은 숫자 계산이 제대로 되지 않을 수 있습니다.

**질문 3. 결측치를 확인한 뒤 바로 모든 행을 삭제하면 안 되는 이유는 무엇인가요?**  
결측치가 있어도 다른 분석에는 사용할 수 있는 데이터일 수 있기 때문에 분석 목적을 먼저 확인해야 합니다.

**질문 4. 같은 상품명이 두 번 등장했다고 해서 바로 중복 데이터라고 판단하기 어려운 이유는 무엇인가요?**  
같은 제목이라도 다른 판본, 개정판, 다른 상품일 수 있기 때문입니다.

**질문 5. AI가 작성한 코드가 오류 없이 실행되었다면 그것만으로 분석이 끝난 것일까요?**  
아닙니다. 실제 출력 결과가 기대한 값인지 다시 확인해야 합니다.


## 제출 전 확인

- [ ] Excel 파일이 정상적으로 로드된다.
- [ ] 데이터 크기와 컬럼을 확인했다.
- [ ] 필요한 컬럼만 선택했다.
- [ ] 컬럼 이름을 정리했다.
- [ ] 결측치를 확인하고 처리했다.
- [ ] 판매가를 숫자형으로 변환했다.
- [ ] 발행일을 날짜형으로 변환했다.
- [ ] 문자열 앞뒤 공백을 정리했다.
- [ ] 중복 데이터를 확인했다.
- [ ] 최종 데이터를 다시 확인했다.
- [ ] `book_bestseller_clean.csv`를 저장했다.
- [ ] Notebook을 처음부터 끝까지 다시 실행했을 때 오류가 없다.
